# Multiple Instance Learning for Normal vs Abnormal Infant Movement

This notebook trains and evaluates attention-based MIL models using `features_notRotated_withQCFlags` and `df_meta_constructed_notRotated_withQCFlags.csv`.

- Label: `final_code_for_ai_str` (`NKD` = 0, `DX` = 1)
- Age strata: `<52` adjusted weeks and `>=52` adjusted weeks
- Clip length: 2700 frames. If a feature array is not already 2700 frames long, the notebook slices from `gma_video_start_1_fnum` to `gma_video_stop_1_fnum`, then pads/truncates only if needed.
- MIL setup: each recording is a bag; overlapping temporal windows are instances; an attention pooling model learns which windows matter.

In [ ]:
from pathlib import Path
import json
import os
import random
import warnings

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "matplotlib-cache"))
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

META_CSV = Path("df_meta_constructed_notRotated_withQCFlags.csv")
FEATURE_DIR = Path("features_notRotated_withQCFlags")
OUT_DIR = Path("mil_notRotated_withQCFlags_results")
OUT_DIR.mkdir(exist_ok=True)

TARGET_FRAMES = 2700
AGE_CUT_WEEKS = 52.0
WINDOW_FRAMES = 90
WINDOW_STRIDE = 45
N_SPLITS = 5
MAX_EPOCHS = 80
PATIENCE = 12
BATCH_SIZE = 1
LR = 1e-3
WEIGHT_DECAY = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

## Load Metadata and Match Feature Files

In [ ]:
meta = pd.read_csv(META_CSV)
meta = meta[meta["final_code_for_ai_str"].isin(["NKD", "DX"])].copy()
meta["adjusted_age_weeks"] = pd.to_numeric(meta["adjusted_age_weeks"], errors="coerce")
meta["gma_video_start_1_fnum"] = pd.to_numeric(meta["gma_video_start_1_fnum"], errors="coerce")
meta["gma_video_stop_1_fnum"] = pd.to_numeric(meta["gma_video_stop_1_fnum"], errors="coerce")
meta["feature_path"] = meta["file"].map(lambda x: FEATURE_DIR / str(x))
meta["has_feature"] = meta["feature_path"].map(Path.exists)
meta["label"] = (meta["final_code_for_ai_str"] == "DX").astype(int)
meta["age_group"] = np.where(meta["adjusted_age_weeks"] < AGE_CUT_WEEKS, "lt52w", "ge52w")

usable = meta[meta["has_feature"] & meta["adjusted_age_weeks"].notna()].reset_index(drop=True)
print(f"metadata rows: {len(meta):,}")
print(f"usable rows with feature arrays: {len(usable):,}")
print("missing feature arrays:", int((~meta["has_feature"]).sum()))
display(usable.groupby(["age_group", "final_code_for_ai_str"]).size().unstack(fill_value=0))

feature_names_path = FEATURE_DIR / "feature_names.json"
feature_names = json.loads(feature_names_path.read_text()) if feature_names_path.exists() else None
if feature_names is not None:
    print(f"feature channels: {len(feature_names)}")

## Clip Extraction and MIL Instance Features

In [ ]:
def load_feature_clip(row, target_frames=TARGET_FRAMES):
    """Return a channels x target_frames array for one recording."""
    arr = np.load(row["feature_path"]).astype(np.float32, copy=False)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D feature array for {row['file']}, got {arr.shape}")

    # Existing feature builders save channels-first arrays. This keeps the notebook robust
    # if a future array arrives frames-first.
    if arr.shape[0] > arr.shape[1]:
        arr = arr.T

    n_channels, n_frames = arr.shape
    if n_frames != target_frames:
        start = row.get("gma_video_start_1_fnum")
        stop = row.get("gma_video_stop_1_fnum")
        if pd.notna(start) and pd.notna(stop):
            start = int(start)
            stop = int(stop)
            if 0 <= start < stop <= n_frames:
                arr = arr[:, start:stop]
            elif 1 <= start < stop <= n_frames:
                arr = arr[:, start - 1:stop - 1]
            else:
                center = n_frames // 2
                left = max(0, center - target_frames // 2)
                arr = arr[:, left:left + target_frames]

    if arr.shape[1] > target_frames:
        arr = arr[:, :target_frames]
    elif arr.shape[1] < target_frames:
        pad = target_frames - arr.shape[1]
        arr = np.pad(arr, ((0, 0), (0, pad)), mode="edge")

    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return arr.astype(np.float32, copy=False)


def window_instance_features(clip, window_frames=WINDOW_FRAMES, stride=WINDOW_STRIDE):
    """Convert channels x frames into a bag of window-level instance vectors."""
    n_channels, n_frames = clip.shape
    starts = list(range(0, max(n_frames - window_frames + 1, 1), stride))
    last = max(0, n_frames - window_frames)
    if starts[-1] != last:
        starts.append(last)

    instances = []
    for start in starts:
        win = clip[:, start:start + window_frames]
        feats = np.concatenate([
            win.mean(axis=1),
            win.std(axis=1),
            np.percentile(win, 10, axis=1),
            np.percentile(win, 50, axis=1),
            np.percentile(win, 90, axis=1),
            np.sqrt(np.mean(np.square(win), axis=1)),
        ])
        instances.append(feats.astype(np.float32))
    return np.stack(instances).astype(np.float32)


def build_bags(df):
    bags = []
    labels = []
    files = []
    for _, row in df.iterrows():
        clip = load_feature_clip(row)
        bags.append(window_instance_features(clip))
        labels.append(int(row["label"]))
        files.append(row["file"])
    return bags, np.asarray(labels, dtype=np.int64), np.asarray(files)


example_clip = load_feature_clip(usable.iloc[0])
example_bag = window_instance_features(example_clip)
print("example clip:", example_clip.shape)
print("example MIL bag:", example_bag.shape, "= windows x instance_features")

## Model Definition

In [ ]:
class BagDataset(Dataset):
    def __init__(self, bags, labels, files):
        self.bags = bags
        self.labels = labels
        self.files = files

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.bags[idx]).float(),
            torch.tensor(self.labels[idx], dtype=torch.float32),
            self.files[idx],
        )


def collate_bags(batch):
    # Batch size 1 keeps variable-size bags simple and makes attention weights easy to inspect.
    bag, label, file = batch[0]
    return bag, label, file


class AttentionMIL(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, attention_dim=64, dropout=0.20):
        super().__init__()
        self.instance_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, attention_dim),
            nn.Tanh(),
            nn.Linear(attention_dim, 1),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, bag):
        h = self.instance_encoder(bag)
        logits_attn = self.attention(h).squeeze(-1)
        attn = torch.softmax(logits_attn, dim=0)
        pooled = torch.sum(attn[:, None] * h, dim=0)
        logit = self.classifier(pooled).squeeze(-1)
        return logit, attn


def fit_scaler(train_bags):
    scaler = StandardScaler()
    scaler.fit(np.vstack(train_bags))
    return scaler


def apply_scaler(bags, scaler):
    return [scaler.transform(bag).astype(np.float32) for bag in bags]

## Training and Evaluation Helpers

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    y_true, y_prob, files, attentions = [], [], [], []
    with torch.no_grad():
        for bag, label, file in loader:
            bag = bag.to(DEVICE)
            logit, attn = model(bag)
            prob = torch.sigmoid(logit).item()
            y_true.append(int(label.item()))
            y_prob.append(prob)
            files.append(file)
            attentions.append(attn.detach().cpu().numpy())
    return np.asarray(y_true), np.asarray(y_prob), np.asarray(files), attentions


def metric_dict(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out = {
        "n": len(y_true),
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "sensitivity_dx": tp / max(tp + fn, 1),
        "specificity_nkd": tn / max(tn + fp, 1),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    }
    out["roc_auc"] = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan
    out["average_precision"] = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan
    return out


def choose_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = [balanced_accuracy_score(y_true, y_prob >= t) for t in thresholds]
    return float(thresholds[int(np.argmax(scores))])


def train_one_fold(train_bags, y_train, train_files, val_bags, y_val, val_files, input_dim):
    train_ds = BagDataset(train_bags, y_train, train_files)
    val_ds = BagDataset(val_bags, y_val, val_files)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_bags)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_bags)

    model = AttentionMIL(input_dim=input_dim).to(DEVICE)
    n_pos = max(int(y_train.sum()), 1)
    n_neg = max(int((y_train == 0).sum()), 1)
    pos_weight = torch.tensor([n_neg / n_pos], device=DEVICE, dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_state = None
    best_val = -np.inf
    stale = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        losses = []
        for bag, label, _ in train_loader:
            bag = bag.to(DEVICE)
            label = label.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logit, _ = model(bag)
            loss = criterion(logit.view(1), label.view(1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            losses.append(loss.item())

        yv, pv, _, _ = evaluate_model(model, val_loader)
        val_auc = roc_auc_score(yv, pv) if len(np.unique(yv)) == 2 else np.nan
        val_bal_acc = balanced_accuracy_score(yv, pv >= 0.5)
        monitor = val_auc if np.isfinite(val_auc) else val_bal_acc
        history.append({"epoch": epoch, "loss": float(np.mean(losses)), "val_auc": val_auc, "val_bal_acc": val_bal_acc})

        if monitor > best_val:
            best_val = monitor
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if stale >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    yv, pv, vf, attn = evaluate_model(model, val_loader)
    threshold = choose_threshold(yv, pv)
    return model, pd.DataFrame(history), yv, pv, vf, attn, threshold

## Cross-Validated Models by Age Group

In [ ]:
all_predictions = []
all_metrics = []
histories = {}

for age_group, group_df in usable.groupby("age_group"):
    group_df = group_df.reset_index(drop=True)
    print(f"\n=== {age_group}: {len(group_df)} recordings ===")
    bags, y, files = build_bags(group_df)
    input_dim = bags[0].shape[1]
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

    for fold, (train_idx, test_idx) in enumerate(skf.split(np.zeros(len(y)), y), start=1):
        train_idx, val_idx = train_test_split(
            train_idx,
            test_size=0.20,
            stratify=y[train_idx],
            random_state=RANDOM_SEED + fold,
        )

        scaler = fit_scaler([bags[i] for i in train_idx])
        train_bags = apply_scaler([bags[i] for i in train_idx], scaler)
        val_bags = apply_scaler([bags[i] for i in val_idx], scaler)
        test_bags = apply_scaler([bags[i] for i in test_idx], scaler)

        model, history, y_val, p_val, _, _, threshold = train_one_fold(
            train_bags, y[train_idx], files[train_idx],
            val_bags, y[val_idx], files[val_idx],
            input_dim=input_dim,
        )
        histories[(age_group, fold)] = history.assign(age_group=age_group, fold=fold)

        test_loader = DataLoader(
            BagDataset(test_bags, y[test_idx], files[test_idx]),
            batch_size=BATCH_SIZE,
            shuffle=False,
            collate_fn=collate_bags,
        )
        y_test, p_test, f_test, attn_test = evaluate_model(model, test_loader)
        metrics = metric_dict(y_test, p_test, threshold=threshold)
        metrics.update({"age_group": age_group, "fold": fold, "val_threshold": threshold})
        all_metrics.append(metrics)

        fold_pred = pd.DataFrame({
            "age_group": age_group,
            "fold": fold,
            "file": f_test,
            "y_true": y_test,
            "p_dx": p_test,
            "threshold": threshold,
            "y_pred": (p_test >= threshold).astype(int),
            "top_attention_window": [int(np.argmax(a)) for a in attn_test],
            "top_attention_start_frame": [int(np.argmax(a) * WINDOW_STRIDE) for a in attn_test],
        })
        all_predictions.append(fold_pred)
        print(f"fold {fold}: auc={metrics['roc_auc']:.3f}, bal_acc={metrics['balanced_accuracy']:.3f}, threshold={threshold:.2f}")

predictions = pd.concat(all_predictions, ignore_index=True)
fold_metrics = pd.DataFrame(all_metrics)
history_df = pd.concat(histories.values(), ignore_index=True)

predictions.to_csv(OUT_DIR / "mil_cv_predictions.csv", index=False)
fold_metrics.to_csv(OUT_DIR / "mil_cv_fold_metrics.csv", index=False)
history_df.to_csv(OUT_DIR / "mil_cv_training_history.csv", index=False)

display(fold_metrics)
display(fold_metrics.groupby("age_group")[["roc_auc", "average_precision", "balanced_accuracy", "sensitivity_dx", "specificity_nkd"]].agg(["mean", "std"]))

## Overall Held-Out Predictions by Age Group

In [ ]:
summary_rows = []
for age_group, df in predictions.groupby("age_group"):
    summary = metric_dict(df["y_true"].to_numpy(), df["p_dx"].to_numpy(), threshold=0.5)
    summary.update({"age_group": age_group, "threshold_source": "fixed_0.5"})
    summary_rows.append(summary)

    # Also evaluate the fold-specific validation threshold already stored in y_pred.
    y_true = df["y_true"].to_numpy()
    y_pred = df["y_pred"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    summary_rows.append({
        "age_group": age_group,
        "threshold_source": "fold_validation_balanced_accuracy",
        "n": len(df),
        "threshold": np.nan,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "sensitivity_dx": tp / max(tp + fn, 1),
        "specificity_nkd": tn / max(tn + fp, 1),
        "roc_auc": roc_auc_score(y_true, df["p_dx"]),
        "average_precision": average_precision_score(y_true, df["p_dx"]),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    })

overall_metrics = pd.DataFrame(summary_rows)
overall_metrics.to_csv(OUT_DIR / "mil_cv_overall_metrics.csv", index=False)
display(overall_metrics)

## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for age_group, df in predictions.groupby("age_group"):
    fpr, tpr, _ = roc_curve(df["y_true"], df["p_dx"])
    auc = roc_auc_score(df["y_true"], df["p_dx"])
    axes[0].plot(fpr, tpr, label=f"{age_group} AUC={auc:.3f}")
    precision, recall, _ = precision_recall_curve(df["y_true"], df["p_dx"])
    ap = average_precision_score(df["y_true"], df["p_dx"])
    axes[1].plot(recall, precision, label=f"{age_group} AP={ap:.3f}")

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC")
axes[0].legend()
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-recall")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "mil_cv_roc_pr_curves.png", dpi=200)
plt.show()

plt.figure(figsize=(8, 4))
for age_group, hist in history_df.groupby("age_group"):
    mean_hist = hist.groupby("epoch")["val_auc"].mean()
    plt.plot(mean_hist.index, mean_hist.values, label=age_group)
plt.xlabel("Epoch")
plt.ylabel("Mean validation ROC AUC")
plt.title("Training Curves")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "mil_cv_training_curves.png", dpi=200)
plt.show()

## Train Final Models on Each Age Stratum

The cross-validation above is the main estimate of performance. This section trains one final model per age stratum using an internal validation split and saves the model weights plus scaler parameters for later inference.

In [ ]:
final_model_dir = OUT_DIR / "final_models"
final_model_dir.mkdir(exist_ok=True)

for age_group, group_df in usable.groupby("age_group"):
    group_df = group_df.reset_index(drop=True)
    bags, y, files = build_bags(group_df)
    train_idx, val_idx = train_test_split(
        np.arange(len(y)),
        test_size=0.20,
        stratify=y,
        random_state=RANDOM_SEED,
    )
    scaler = fit_scaler([bags[i] for i in train_idx])
    train_bags = apply_scaler([bags[i] for i in train_idx], scaler)
    val_bags = apply_scaler([bags[i] for i in val_idx], scaler)
    model, history, y_val, p_val, val_files, val_attn, threshold = train_one_fold(
        train_bags, y[train_idx], files[train_idx],
        val_bags, y[val_idx], files[val_idx],
        input_dim=bags[0].shape[1],
    )
    payload = {
        "age_group": age_group,
        "model_state_dict": model.state_dict(),
        "input_dim": bags[0].shape[1],
        "window_frames": WINDOW_FRAMES,
        "window_stride": WINDOW_STRIDE,
        "target_frames": TARGET_FRAMES,
        "threshold": threshold,
        "scaler_mean": scaler.mean_.astype(np.float32),
        "scaler_scale": scaler.scale_.astype(np.float32),
        "feature_names": feature_names,
    }
    torch.save(payload, final_model_dir / f"attention_mil_{age_group}.pt")
    history.assign(age_group=age_group).to_csv(final_model_dir / f"attention_mil_{age_group}_history.csv", index=False)
    print(age_group, metric_dict(y_val, p_val, threshold=threshold))